# P105 — Product Cannibalization & Demand Shift Analysis
## Phase 5: Cannibalization & Demand Shift Deep Dive

**Objective:** Quantify the cannibalization effect, measure demand shift, and analyze customer migration patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
os.makedirs('../Images/Phase5', exist_ok=True)

LAUNCH_DATE = pd.Timestamp('2024-06-01')
LAUNCHED_PRODUCTS = ['P1', 'P2', 'P3', 'P4', 'P5']

## Load Cleaned Data

In [ ]:
df = pd.read_csv("../Data/cannibalization_cleaned.csv")
df['Date'] = pd.to_datetime(df['Date'])
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

## 1. Product Sales — Before vs After Launch (Waterfall View)

In [ ]:
before = df[df['Period_Flag'] == 'Before_Launch'].groupby('Product_ID')['Sales'].mean()
after  = df[df['Period_Flag'] == 'After_Launch'].groupby('Product_ID')['Sales'].mean()

comparison = pd.DataFrame({
    'Before': before,
    'After': after
}).fillna(0).round(2)
comparison['Change'] = comparison['After'] - comparison['Before']
comparison['Change_Pct'] = ((comparison['Change'] / comparison['Before']) * 100).replace([np.inf, -np.inf], np.nan).round(2)
comparison.sort_values('Change', inplace=True)

plt.figure(figsize=(12, 7))
colors = ['green' if x > 0 else 'red' for x in comparison['Change']]
plt.barh(comparison.index, comparison['Change'], color=colors)
plt.axvline(0, color='gray', linewidth=1)
plt.title('Sales Change per Product — After vs Before Launch', fontweight='bold')
plt.xlabel('Change in Avg Sales (Units)')
plt.tight_layout()
plt.savefig('../Images/Phase5/01_sales_change_waterfall.png', bbox_inches='tight')
plt.show()

print("\nProduct Sales Comparison:")
print(comparison.to_string())

## 2. Revenue Shift Analysis

In [ ]:
rev_before = df[df['Period_Flag'] == 'Before_Launch'].groupby('Product_ID')['Revenue'].sum()
rev_after  = df[df['Period_Flag'] == 'After_Launch'].groupby('Product_ID')['Revenue'].sum()

rev_comp = pd.DataFrame({
    'Rev_Before': rev_before,
    'Rev_After': rev_after
}).fillna(0)
rev_comp['Rev_Change'] = rev_comp['Rev_After'] - rev_comp['Rev_Before']
rev_comp['Rev_Change_Pct'] = ((rev_comp['Rev_Change'] / rev_comp['Rev_Before']) * 100).replace([np.inf, -np.inf], np.nan).round(2)
rev_comp.sort_values('Rev_Change', inplace=True)

plt.figure(figsize=(12, 7))
colors = ['green' if x > 0 else 'red' for x in rev_comp['Rev_Change']]
plt.barh(rev_comp.index, rev_comp['Rev_Change'], color=colors)
plt.axvline(0, color='gray', linewidth=1)
plt.title('Revenue Change per Product — After vs Before Launch', fontweight='bold')
plt.xlabel('Revenue Change')
plt.tight_layout()
plt.savefig('../Images/Phase5/02_revenue_shift.png', bbox_inches='tight')
plt.show()

## 3. Cannibalization Rate Calculation

In [ ]:
# P6 = cannibalized product in Group G2
# P4, P5 = new launched products in Group G2

# Normalize for different time period lengths
months_before = df[df['Period_Flag'] == 'Before_Launch']['Year_Month'].nunique()
months_after  = df[df['Period_Flag'] == 'After_Launch']['Year_Month'].nunique()

# P6 loss
p6_before = df[(df['Product_ID'] == 'P6') & (df['Period_Flag'] == 'Before_Launch')]['Sales'].sum()
p6_after  = df[(df['Product_ID'] == 'P6') & (df['Period_Flag'] == 'After_Launch')]['Sales'].sum()
p6_norm_before = (p6_before / months_before) * months_after
p6_loss = p6_norm_before - p6_after

# P4+P5 gain
p4p5_gain = df[df['Product_ID'].isin(['P4', 'P5']) & (df['Period_Flag'] == 'After_Launch')]['Sales'].sum()

# Cannibalization Rate
cann_rate = (p6_loss / p4p5_gain * 100) if p4p5_gain > 0 else 0

print("=" * 50)
print("  CANNIBALIZATION RATE CALCULATION")
print("=" * 50)
print(f"  Months Before Launch     : {months_before}")
print(f"  Months After Launch      : {months_after}")
print(f"  P6 Sales Before (norm)   : {p6_norm_before:,.0f}")
print(f"  P6 Sales After           : {p6_after:,.0f}")
print(f"  P6 Sales Loss            : {p6_loss:,.0f}")
print(f"  P4+P5 Sales Gained       : {p4p5_gain:,.0f}")
print(f"  Cannibalization Rate     : {cann_rate:.1f}%")
print("=" * 50)

## 4. Demand Shift Timeline — P6 vs P4/P5

In [ ]:
prod_monthly = df.groupby(['Year_Month', 'Product_ID'])['Sales'].sum().reset_index()
prod_monthly['Date'] = pd.to_datetime(prod_monthly['Year_Month'])

fig, ax = plt.subplots(figsize=(14, 6))

# P6
p6 = prod_monthly[prod_monthly['Product_ID'] == 'P6'].sort_values('Date')
ax.plot(p6['Date'], p6['Sales'], 'r-o', linewidth=2.5, markersize=5, label='P6 (Cannibalized)')
ax.fill_between(p6['Date'], p6['Sales'], alpha=0.1, color='red')

# P4, P5
for pid, color in [('P4', 'blue'), ('P5', 'green')]:
    pdata = prod_monthly[prod_monthly['Product_ID'] == pid].sort_values('Date')
    ax.plot(pdata['Date'], pdata['Sales'], '-o', color=color, linewidth=2, markersize=4, label=f'{pid} (Launched)')

ax.axvline(LAUNCH_DATE, color='gray', linestyle='--', linewidth=2, label='Launch Date')
ax.set_title('Demand Shift in Group G2 — P6 Decline vs P4/P5 Rise', fontweight='bold', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Total Sales (Units)')
ax.legend(fontsize=10)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../Images/Phase5/03_demand_shift_timeline.png', bbox_inches='tight')
plt.show()

## 5. Customer Migration Analysis

In [ ]:
# Customers who bought P6 before launch
cust_p6_before = set(df[(df['Product_ID'] == 'P6') & (df['Period_Flag'] == 'Before_Launch')]['Customer_ID'])

# Customers who bought P4/P5 after launch
cust_new_after = set(df[(df['Product_ID'].isin(['P4', 'P5'])) & (df['Period_Flag'] == 'After_Launch')]['Customer_ID'])

# Customers who stayed with P6 after launch
cust_p6_after = set(df[(df['Product_ID'] == 'P6') & (df['Period_Flag'] == 'After_Launch')]['Customer_ID'])

# Switchers = bought P6 before AND bought P4/P5 after
switchers = cust_p6_before & cust_new_after
stayed = cust_p6_before & cust_p6_after
lost = cust_p6_before - cust_p6_after - switchers

switching_rate = len(switchers) / len(cust_p6_before) * 100 if cust_p6_before else 0

print("=" * 50)
print("  CUSTOMER MIGRATION ANALYSIS")
print("=" * 50)
print(f"  P6 Customers (Before)    : {len(cust_p6_before):,}")
print(f"  Switched to P4/P5        : {len(switchers):,}")
print(f"  Stayed with P6           : {len(stayed):,}")
print(f"  Lost (neither)           : {len(lost):,}")
print(f"  Switching Rate           : {switching_rate:.1f}%")
print("=" * 50)

# Pie chart
labels = ['Switched to P4/P5', 'Stayed with P6', 'Lost']
sizes = [len(switchers), len(stayed), len(lost)]
colors_pie = ['#ff6b6b', '#51cf66', '#ffd43b']

plt.figure(figsize=(7, 7))
plt.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.1f%%',
        startangle=140, textprops={'fontsize': 11})
plt.title('P6 Customer Migration After Launch', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../Images/Phase5/04_customer_migration.png', bbox_inches='tight')
plt.show()

## 6. Affected Groups Analysis — G1 vs G2

In [ ]:
affected = df[df['Product_Group'].isin(['G1', 'G2'])]
grp_period = affected.groupby(['Product_Group', 'Product_ID', 'Period_Flag'])['Sales'].mean().unstack()
grp_period.columns = ['After', 'Before']
grp_period['Change_Pct'] = ((grp_period['After'] - grp_period['Before']) / grp_period['Before'] * 100).round(2)

print("Affected Groups — Before vs After:")
print(grp_period.to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# G1
g1 = grp_period.loc['G1']
colors = ['green' if x > 0 else 'red' for x in g1['Change_Pct']]
ax1.barh(g1.index, g1['Change_Pct'], color=colors)
ax1.axvline(0, color='gray')
ax1.set_title('G1 — Sales Change % (P1, P2, P3)', fontweight='bold')
ax1.set_xlabel('Change %')

# G2
g2 = grp_period.loc['G2']
colors = ['green' if x > 0 else 'red' for x in g2['Change_Pct']]
ax2.barh(g2.index, g2['Change_Pct'], color=colors)
ax2.axvline(0, color='gray')
ax2.set_title('G2 — Sales Change % (P4, P5, P6)', fontweight='bold')
ax2.set_xlabel('Change %')

plt.tight_layout()
plt.savefig('../Images/Phase5/05_affected_groups.png', bbox_inches='tight')
plt.show()

## 7. Cannibalization Summary Table

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Launch Date',
        'Launched Products (G1)',
        'Launched Products (G2)',
        'Cannibalized Product',
        'Cannibalization Rate',
        'Customer Switching Rate',
        'P6 Sales Loss (normalized)',
        'P4+P5 Sales Gained',
        'Demand Shift Records'
    ],
    'Value': [
        '2024-06-01',
        'P1, P2, P3',
        'P4, P5',
        'P6 (Beverages, Group G2)',
        f'{cann_rate:.1f}%',
        f'{switching_rate:.1f}%',
        f'{p6_loss:,.0f} units',
        f'{p4p5_gain:,.0f} units',
        f"{df['Demand_Shift_Flag'].sum():,}"
    ]
})

print("=" * 50)
print("  CANNIBALIZATION SUMMARY")
print("=" * 50)
print(summary.to_string(index=False))

## Phase 5 Summary

**Key Findings:**
- P6 experienced significant sales decline after the launch of P4 and P5 in the same group (G2)
- The cannibalization rate shows what portion of new product sales came at P6's expense
- A meaningful percentage of P6's former customers switched to the new products
- Group G1 (P1, P2, P3) and G2 (P4, P5, P6) both show post-launch dynamics
- Demand shift is confirmed: P6's loss correlates with P4/P5's gain

**Next:** Phase 6 — Customer, Pricing, Marketing & Inventory Analysis